# Evaluate LoRA × GPT-2 Medium on E2E NLG Challenge

This notebook reproduces the GPT-2 Medium + LoRA row of **Table 3** from [Hu et al., 2021](https://arxiv.org/abs/2106.09685). It expects:
- `code.zip` — the project source (same one used for training)
- `lora_best.pt` — the LoRA adapter checkpoint saved by `Trainer` during the E2E training run (~1.5 MB)

**Pipeline:** load `gpt2-medium`, inject + load the LoRA adapter, merge into base for fast inference, beam-search-decode the test split (~4.7k unique MRs at beam=10), score with the 5 metrics (BLEU / NIST / METEOR / ROUGE-L / CIDEr), and render a comparison table against the paper.

**Runtime budget:** ~30–45 min on T4 for the generation pass; <1 min for scoring.

In [ ]:
!nvidia-smi

## 1. Upload `code.zip`

In [ ]:
!git clone https://github.com/austinlu2005/myLoRA.git /content/myLoRA
%cd /content/myLoRA/code

## 2. Upload `lora_best.pt`

The adapter file from your training run. It's small (~1.5 MB).

In [ ]:
uploaded = files.upload()  # drag-drop lora_best.pt
lora_name = next(iter(uploaded))
import shutil
LORA_PATH = "/content/lora_best.pt"
shutil.move(lora_name, LORA_PATH)
print(f"Adapter at: {LORA_PATH}")

## 3. Install dependencies

`datasets<4.0` pin is no longer required — `dataloaders/e2e_nlg.py` now uses the auto-generated parquet mirror via `revision="refs/convert/parquet"`.

Pip metric stack: `nltk` (BLEU / NIST / METEOR), `rouge_score`, `pycocoevalcap` (CIDEr), `sacrebleu`.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q nltk rouge_score pycocoevalcap sacrebleu

## 4. (Optional) Clone the official scorer for paper-exact numbers

The pip stack above produces numbers within ~0.1–0.5 BLEU of paper. For paper-exact scoring, clone tuetschek/e2e-metrics — it bundles the Perl `mteval-v13a` for BLEU/NIST and the Java `meteor-1.5.jar` for METEOR. Colab runtimes have JDK + Perl preinstalled so this just works.

Skip this cell if you don't need exact paper match.

In [ ]:
USE_OFFICIAL_SCORER = False  # flip to True if you want paper-exact metrics

E2E_METRICS_DIR = None
if USE_OFFICIAL_SCORER:
    !git clone -q https://github.com/tuetschek/e2e-metrics.git /content/e2e-metrics
    !pip install -q --user -r /content/e2e-metrics/requirements.txt
    E2E_METRICS_DIR = "/content/e2e-metrics"

## 5. Run the eval driver

Beam=10, length_penalty=0.9, no_repeat_ngram=4 — paper-aligned generation hyperparameters. `--max-eval 0` runs the full ~4.7k unique MRs. Set `--max-eval 200` for a quick smoke test (~2 min).

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "scripts/evaluate_e2e.py",
    "--lora-path", LORA_PATH,
    "--model-name", "gpt2-medium",
    "--rank", "4", "--alpha", "32", "--dropout", "0.1",
    "--output-dir", "/content/e2e_eval",
    "--beam-size", "10",
    "--length-penalty", "0.9",
    "--no-repeat-ngram-size", "4",
    "--max-new-tokens", "80",
    "--max-eval", "0",  # 0 = full test set; set e.g. 200 for a quick smoke test
]
if E2E_METRICS_DIR:
    cmd += ["--e2e-metrics", E2E_METRICS_DIR]

# Stream stdout live so we see the tqdm progress bar
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
assert process.returncode == 0, f"evaluate_e2e.py exited with {process.returncode}"

## 6. Render the paper-comparison row

Pulls `results.json` from the eval run and produces a single-row Table 3 comparison.

In [ ]:
import json
import pandas as pd

with open("/content/e2e_eval/results.json") as f:
    res = json.load(f)

ours = res["metrics"]
paper = res["paper"]

# BLEU/METEOR/ROUGE-L are reported as percentages in the paper; NIST/CIDEr as raw
def fmt(val, key):
    if val is None or (isinstance(val, float) and val != val):
        return "–"
    scale = 100 if key in ("BLEU", "METEOR", "ROUGE_L") else 1
    return f"{val * scale:.2f}"

rows = [
    {"system": "GPT-2 M (LoRA, paper Table 3)", **{k: fmt(paper[k], k) for k in ("BLEU", "NIST", "METEOR", "ROUGE_L", "CIDEr")}},
    {"system": "GPT-2 M (LoRA, ours)",          **{k: fmt(ours.get(k), k) for k in ("BLEU", "NIST", "METEOR", "ROUGE_L", "CIDEr")}},
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save the comparison table for the report
df.to_csv("/content/e2e_eval/comparison_table.csv", index=False)
print("\nSaved /content/e2e_eval/comparison_table.csv")

## 7. (Optional) Inspect a few generations

In [ ]:
with open("/content/e2e_eval/predictions.txt") as f:
    preds = f.read().splitlines()
with open("/content/e2e_eval/references.txt") as f:
    raw = f.read()
ref_groups = [g.splitlines() for g in raw.split("\n\n") if g.strip()]

for i in (0, 1, 2, 100, 500):
    if i >= len(preds):
        break
    print(f"--- example {i} ---")
    print(f"GEN : {preds[i]}")
    for j, r in enumerate(ref_groups[i][:3]):
        print(f"REF{j}: {r}")
    print()

## 8. Persist results to Drive

`/content` is wiped when the runtime ends. Copy the eval artifacts to Drive so the report can pull from them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/e2e_eval /content/drive/MyDrive/lora_runs_gpt2_e2e_eval

## Notes

- **Default scorer is the pip stack** (`nltk` BLEU/NIST/METEOR, `rouge_score`, `pycocoevalcap` CIDEr). Numbers will be within ~0.1–0.5 BLEU of paper. Set `USE_OFFICIAL_SCORER = True` in cell 4 for paper-exact scoring via `tuetschek/e2e-metrics`.
- **Generation params** (`beam=10, length_penalty=0.9, no_repeat_ngram=4`) match the LoRA paper's E2E settings. Tuning these is fair game for ablations but not for the headline number.
- **`--max-eval 200`** is a fast smoke test (<2 min) — useful before committing to the full ~30–45 min run.
- **Re-scoring without re-generating.** `predictions.txt` and `references.txt` persist in `/content/e2e_eval`. To re-score with a different backend, point `score_e2e_official` (or `score_e2e_pip`) at those files directly without re-running generation.